In [3]:
# coding: utf-8
import arcpy
import os
from collections import defaultdict

# ================= 核心配置参数 =================
target_shp = r"F:\linfen\patch\子项目\分类\矿山.shp"
county_shp = r"F:\linfen\boundary\临汾市区县.shp"
output_shp = r"F:\linfen\分类\矿山_结果.shp"

# ⚠️ 【非常重要！】请打开“临汾市投影.shp”，看属性表里代表“县名”的字段叫什么。
# 例如：叫 "NAME"、"Name"、"县名"、"县"。务必把这里改成真实的字段名。
county_name_field = "MERGE_SRC"  # 默认假设字段为Name，如不对请修改

# 距离参数（单位：米）
buffer_dist = 2000
merge_dist = 5000     # 5km 内合并
discard_dist = 25000  # 25km 外的矿山块丢弃

# 设置环境
arcpy.env.overwriteOutput = True
# ============================================

class DSU:
    """并查集类，用于合并连通的多边形"""
    def __init__(self):
        self.parent = {}
    
    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry:
            self.parent[ry] = rx

print("-" * 30)
print("开始执行地理处理，请稍候...")

try:
    # 1. 2公里缓冲区并融合
    print("1. 正在生成 2km 缓冲区...")
    buffer_fc = os.path.join(arcpy.env.scratchGDB, "buffer_2km")
    arcpy.analysis.Buffer(target_shp, buffer_fc, f"{buffer_dist} Meters", "FULL", "ROUND", "ALL", None)
    
    # 安全校验：检查是否生成了有效的缓冲区要素
    count = int(arcpy.GetCount_management(buffer_fc).getOutput(0))
    if count == 0:
        raise Exception("错误：缓冲区生成了空数据！请检查您的 矿山.shp 是否位于 临汾市投影.shp 的坐标系内。")
    print(f"   已生成 {count} 个缓冲要素。")

    # 2. 与县界相交切割
    print("2. 正在与县界进行相交切割...")
    intersect_fc = os.path.join(arcpy.env.scratchGDB, "intersect_buf")
    arcpy.analysis.Intersect([buffer_fc, county_shp], intersect_fc, "ALL")

    # 3. 按县名字段融合成一个整体多边形（保护不跨县）
    print("3. 正在按县进行融合...")
    dissolved_fc = os.path.join(arcpy.env.scratchGDB, "dissolved_by_county")
    arcpy.management.Dissolve(intersect_fc, dissolved_fc, [county_name_field], None, "MULTI_PART", "DISSOLVE_LINES")

    # 4. 生成最小凸包 (Minimum Convex Hull)
    print("4. 正在生成各县域内的最小凸包...")
    convex_hulls_fc = os.path.join(arcpy.env.scratchGDB, "convex_hulls")
    # "LIST" 分组模式保证每个县生成唯一的凸包
    arcpy.management.MinimumBoundingGeometry(dissolved_fc, convex_hulls_fc, "CONVEX_HULL", "LIST", county_name_field, "NO_MBG_FIELDS")

    # 5. 计算各凸包之间的最近距离
    print("5. 正在计算各凸包的邻域距离...")
    geoms = {}
    with arcpy.da.SearchCursor(convex_hulls_fc, ["FID", "SHAPE@"]) as cursor:
        for row in cursor:
            geoms[row[0]] = row[1]
    
    fids = list(geoms.keys())
    if not fids:
        raise Exception("错误：未生成任何凸包，请检查矿山是否在县界内。")
    
    dsu = DSU()
    min_dists = {fid: float('inf') for fid in fids}
    
    for i in range(len(fids)):
        for j in range(i + 1, len(fids)):
            fid_i, fid_j = fids[i], fids[j]
            # 计算两凸包间的最短几何距离
            d = geoms[fid_i].distanceTo(geoms[fid_j])
            
            if d < min_dists[fid_i]:
                min_dists[fid_i] = d
            if d < min_dists[fid_j]:
                min_dists[fid_j] = d
            
            # 若距离 <= 5km，标记为合并组
            if d <= merge_dist:
                dsu.union(fid_i, fid_j)
    
    # 6. 应用丢弃规则（超过25km的孤岛剔除）和合并规则
    print("6. 应用筛选和聚合规则...")
    keep_fids = set()
    for fid in fids:
        if len(fids) == 1:
            keep_fids.add(fid)
        elif min_dists[fid] <= discard_dist:
            keep_fids.add(fid)
        # 超过 discard_dist 的 fid 未加入 keep_fids，相当于被丢弃

    # 7. 分组：找出需要合并的组
    final_groups = defaultdict(list)
    for fid in keep_fids:
        root = dsu.find(fid)
        final_groups[root].append(fid)

    # 8. 导出最终结果
    print("7. 正在导出最终结果要素...")
    
    # 自动创建结果输出文件夹（防止路径不存在报错）
    out_dir = os.path.dirname(output_shp)
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)

    if arcpy.Exists(output_shp):
        arcpy.management.Delete(output_shp)
    
    spatial_ref = arcpy.Describe(convex_hulls_fc).spatialReference
    arcpy.management.CreateFeatureclass(
        os.path.dirname(output_shp), 
        os.path.basename(output_shp), 
        "POLYGON", 
        spatial_reference=spatial_ref
    )

    with arcpy.da.InsertCursor(output_shp, ["SHAPE@"]) as insert_cursor:
        for root, group_fids in final_groups.items():
            if len(group_fids) == 1:
                # 单个保留的凸包
                insert_cursor.insertRow([geoms[group_fids[0]]])
            else:
                # 多个相邻凸包融合成一个
                fid_str = ",".join([str(fid) for fid in group_fids])
                temp_layer = "in_memory/temp_sel_layer"
                # 注意：虽然这里用了 in_memory，但只是做一个临时查询，后续马上删掉，不会引起崩溃
                arcpy.management.MakeFeatureLayer(convex_hulls_fc, temp_layer, f"FID IN ({fid_str})")
                temp_diss = os.path.join(arcpy.env.scratchGDB, "temp_diss_group")
                arcpy.management.Dissolve(temp_layer, temp_diss)
                
                with arcpy.da.SearchCursor(temp_diss, ["SHAPE@"]) as cur:
                    for row in cur:
                        insert_cursor.insertRow([row[0]])
                
                # 清理临时数据
                arcpy.management.Delete(temp_layer)
                arcpy.management.Delete(temp_diss)

    print("=" * 30)
    print(f"✅ 大功告成！最终结果已保存至：{output_shp}")

except Exception as e:
    print("=" * 30)
    print("❌ 程序执行出错！")
    print(f"错误详情: {e}")
    print("\n💡 解决建议：")
    print("1. 检查 '矿山.shp' 和 '临汾市投影.shp' 坐标系是否统一（必须是投影坐标系，如 UTM 或 CGCS2000）；")
    print("2. 检查 'county_name_field' 变量，是否和 '临汾市投影.shp' 属性表中的县名实际字段名一致。")

------------------------------
开始执行地理处理，请稍候...
1. 正在生成 2km 缓冲区...
   已生成 1 个缓冲要素。
2. 正在与县界进行相交切割...
3. 正在按县进行融合...
4. 正在生成各县域内的最小凸包...
5. 正在计算各凸包的邻域距离...
❌ 程序执行出错！
错误详情: Cannot find field 'FID'

💡 解决建议：
1. 检查 '矿山.shp' 和 '临汾市投影.shp' 坐标系是否统一（必须是投影坐标系，如 UTM 或 CGCS2000）；
2. 检查 'county_name_field' 变量，是否和 '临汾市投影.shp' 属性表中的县名实际字段名一致。
